# **Proyecto NLP**

* Aplicar un flujo básico de procesamiento de lenguaje natural (NLP) para resolver un problema de clasificación.
* Objetivo: Queremos implementar un sistema que sea capaz de detectar automáticamente si una página web contiene spam o no basándonos en su URL.

* El proyecto incluye:  
        - Análisis exploratorio de datos (EDA)  
        - Extracción de características de URLs  
        - Vectorización de texto  
        - Entrenamiento de modelos de clasificación

**APUNTES:** 
* Info del modelo: https://bbycroft.net/llm

## **Imports**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
import joblib

## **EDA**

* ¿Qué palabras aportan al objetivo de analizar si una pagina es spam o no? --> Investigar y analizar data

### **Lectura e información de los datos**

In [30]:
df = pd.read_csv("https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv")
df.head()


,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


> **Observaciones:**  
> * Columnas:  
>           - url = texto  
>           - is_spam = etiqueta (¿el url pertenece a un spam? ¿si o no?)

In [ ]:
df.shape

### **Limpieza de datos**

In [ ]:
df.info()


In [ ]:
df.isnull().sum()

### **Balance de clases**

* Permite detectar desbalanceo de clases que suele ser algo común en problemas de spam.

In [ ]:
sns.countplot(x="is_spam", data=df)
plt.title("Distribución de URLs Spam vs No Spam")
plt.show()

> **Observaciones:**

### **Future engineering (Análisis de las variables)**

* **Importante:** Extraer patrones estructurales de la URL.

* Las URLs de spam suelen ser más largas para ocultar palabras sospechosas.

#### 1. **Conteo de data**

In [ ]:
df["url_length"] = df["url"].apply(len)

In [ ]:
sns.boxplot(x="is_spam", y="url_length", data=df)

* **Número de dígitos:** Las url Spam usa muchos números.

In [ ]:
df["digits_count"] = df["url"].apply(lambda x: sum(c.isdigit() for c in x))

* **Número de caracteres especiales:** Las URLs maliciosas utilizan más símbolos y separadores para disfrazar el enlace.

In [ ]:
df["special_chars"] = df["url"].apply(lambda x: len(re.findall(r"[^\w\s]", x)))

* **Número de subdirectorios:** Las URLs de spam suelen tener muchos subdirectorios falsos.

In [ ]:
df["slash_count"] = df["url"].apply(lambda x: x.count("/"))

#### 2. **Detección de palabras sosprechosas**

Palabras sospechosas:

* free
* login
* verify
* account
* update
* secure
* bank
* bonus
* win
* credit
* offer
* click* 

In [ ]:
spam_words = ["free","login","verify","account","update","bonus","bank","secure","win"]

def contains_spam_words(url):
    url = url.lower()
    return sum(word in url for word in spam_words)

df["spam_word_count"] = df["url"].apply(contains_spam_words)

## **Procesamiento de texto**

In [ ]:
# tokenizacion -DUDA que eS?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["url"])

* **Palabras comunes**

In [ ]:
import numpy as np

words = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1

freq = pd.DataFrame({
    "word": words,
    "count": counts
}).sort_values("count", ascending=False)

freq.head(20)

* **Palabras comunes en spam** : Permite identificar tokens asociados al spam. DUDA--> Como funciona?

In [ ]:
spam_urls = df[df["is_spam"]==1]["url"]

vectorizer = CountVectorizer(stop_words="english")
X_spam = vectorizer.fit_transform(spam_urls)

words = vectorizer.get_feature_names_out()
counts = X_spam.sum(axis=0).A1

spam_freq = pd.DataFrame({
    "word": words,
    "count": counts
}).sort_values("count", ascending=False)

spam_freq.head(20)

* **Conversión del texto de la url**

In [31]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-zA-Z]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df["url_limpio"] = df["url"].apply(limpiar_texto)

### **Declarar variables**

In [32]:
X_text = df["url_limpio"]
y = df["is_spam"]

#### **Vectorizar el texto**

No interpreta el siginificado del texto. Si cambia le orden de palabras cambia significado del texto peo el modelo no toma la relacion entre palabras. 

De esta manera se puede perder informacion por el camino.


In [38]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(X_text)
print(vectorizer.get_feature_names_out())
print(X.toarray())

['aa' 'aab' 'aaron' ... 'zwift' 'zwn' 'zyguxzjc']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
vect_df = pd.DataFrame(X.todense(), columns=vectorizer.get_feature_names_out())
vect_df["is_spam"] = y # vect_df["Resultado Esperado"] = df["is_spam"]

vect_df.head()

,aa,aab,aaron,ab,abacus,abandoned,abba,abbott,abbreviated,abc,...,zskl,ztz,zuck,zuckerberg,zuihitsu,zulalimtm,zwift,zwn,zyguxzjc,is_spam
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,False
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True


#### **Transformers**

Modelo entrenado que tiene en cuenta el orden y la relacion de las palabras.

In [ ]:
# insert code

### **Split data train - test**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **ML**

### **Naive Bayes**

* **Crear y entrenar del modelo**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

* **Predicciones:**

In [ ]:
y_pred = model.predict(X_test)

* **Evaluación del modelo (Accuracy)**:

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

* **Test data vs Data predicha**

In [ ]:
resultados = pd.DataFrame({    "Real": y_test,"Prediccion": y_pred})
resultados.head()

### **Modelo SVM**

* **Crear y entrenar el modelo**

In [ ]:
svm_model = SVC()

In [ ]:
svm_model.fit(X_train, y_train)

* **Predicciones:**

In [ ]:
y_pred_svm = svm_model.predict(X_test)

* **Evaluación del modelo (Accuracy)**:

In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("Accuracy SVM:", accuracy_svm)

* **Métricas:**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

### **Optimización del modelo (GridSearch)**

In [ ]:
param_grid = {'C': [0.1, 1, 10],'kernel': ['linear', 'rbf'],'gamma': ['scale', 'auto']}
grid = GridSearchCV(SVC(),param_grid,cv=5,scoring='accuracy',n_jobs=-1)
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor score:", grid.best_score_)

In [ ]:
best_svm = grid.best_estimator_
y_pred_best = best_svm.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print("Accuracy optimizado:", accuracy_best)

### **Guardado del modelo**

In [ ]:
joblib.dump(best_svm, "svm_spam_model.pkl")

In [ ]:
modelo_cargado = joblib.load("svm_spam_model.pkl")

## **Observaciones finales:**


* Preparación de los datos: limpieza del texto, transformando todas las URLs a minúsculas y eliminando símbolos para separar las palabras que componen cada dirección los modelos de machine learning no pueden trabajar directamente con texto sin procesar.
* División del dataset en conjuntos de entrenamiento y prueba para poder evaluar el rendimiento del modelo de forma objetiva. 
* Se entrena también un modelo de clasificación Naive Bayes, que es especialmente adecuado para problemas de texto debido a su simplicidad y buen rendimiento en tareas de clasificación. Y se evalua el modelo utilizando el conjunto de prueba y se obtuvo una métrica de precisión (accuracy) que permite estimar qué tan bien el modelo es capaz de distinguir entre URLs spam y no spam.